#### GPU warm/col time feature comparison
Context: 
For every function, average GPU warm execution times and average GPU cold execution times are benchmark values used as features in RF prediction model. During model training, the 2 features were available from `worker_function_benchmarks.json`. However, this information(avg warm/cold exection averaged over cerrain runs) is not available in iluvatar library to which the model is integrated to. Through cmap, avg cold/warm execution times are available but those are moving averages.

This notebook aims to analyse the difference between the feature values at fqdn level and understand its impact on the model predictions

In [1]:
import numpy as np
import pandas as pd

In [3]:
import sys
import os
sys.path.append(os.path.abspath('../../..'))

In [5]:
from src.scripts.utils import read_log_as_csv, get_workerlog_all_paths

In [6]:
paths = get_workerlog_all_paths('/Users/akshaykishan/latest_data/')

In [8]:
sample_data = read_log_as_csv(paths[0])

In [10]:
sample_data.columns

Index(['timestamp', 'level', 'target', 'message', 'tid', 'gpus', 'used_mem',
       'total_mem', 'num_containers', 'num_running_funcs', 'queue_info',
       'snapshotter', 'image', 'command', 'cpu_util', 'gpu_util', 'name',
       'function_name', 'function_version', 'fqdn', 'address', 'span.tid',
       'span.name', 'image_name', 'fn_queue_len', 'fn_in_flight',
       'total_queue_len', 'total_in_flight', 'active_flows', 'raw_est',
       'error', 'kf_est', 'll_present', 'll_credit', 'physical_state',
       'is_disparity', 'is_warm_gpu', 'mqfq_est', 'gpu_est', 'gpu_adj_est',
       'gpu_est_err', 'cpu_est', 'cpu_exec', 'gpu_est_total', 'cpu_est_total',
       'gpu_warm', 'gpu_cold', 'total_load', 'queue', 'old_state', 'new_state',
       'start_vt', 'queue_len', 'uuid', 'cid', 'container_id', 'threads',
       'memory', 'gpu_load', 'opp_cost', 'state', 'insert_time', 'remove_time',
       'pot_creds', 'e2etime', 'compute', 'uncapped', 'cap', 'acc_pot_credits',
       'fn_slowdown', '

In [26]:
sample_data[sample_data['message'].str.contains('Landlord Credit')][['fqdn', 'tid', 'message', 'gpu_est_total', 'gpu_warm', 'gpu_cold', 'compute']]['tid'].nunique()

2628

In [ ]:
sample_data[sample_data['']]

In [27]:
gpu_tids = sample_data[sample_data['compute']=='GPU']['tid'].unique()
cpu_tids = sample_data[sample_data['compute']=='CPU']['tid'].unique()
gpu_tids.shape[0], cpu_tids.shape[0], 

(878, 1745)

In [35]:
fqdn_data = sample_data[(sample_data['message'].str.contains('Landlord Credit')) & (sample_data['tid'].isin(gpu_tids))][['fqdn', 'tid', 'message', 'gpu_est_total', 'gpu_warm', 'gpu_cold']]

In [36]:
sample_data[(sample_data['message'].str.contains('Landlord Credit')) & (sample_data['tid'].isin(cpu_tids))][['fqdn', 'tid', 'message', 'gpu_est_total', 'gpu_warm', 'gpu_cold', 'compute']]

,fqdn,tid,message,gpu_est_total,gpu_warm,gpu_cold,compute
413,ckn_resnet50-3-0.0.1,49c13ada35d25a8d469c4e8fdc941e57,Landlord Credit,NaN,0.000000,0.000000,NaN
602,ckn_resnet50-3-0.0.1,3726bbe9747f29ebf37e4b4fe5aa7334,Landlord Credit,0.000000,0.000000,0.000000,NaN
676,pyhpc-eos-3-0.0.1,87c951ae0d1ca6f2365902cdb2fda134,Landlord Credit,1.293562,0.006944,3.775113,NaN
679,pyhpc-eos-3-0.0.1,87c951ae0d1ca6f2365902cdb2fda134,Landlord Credit,1.293562,0.006944,3.775113,NaN
717,pyhpc-eos-3-0.0.1,aab65e288c8698dca3674f071f230310,Landlord Credit,2.096076,0.006944,4.107867,NaN
...,...,...,...,...,...,...,...
44788,pyhpc-eos-3-0.0.1,8b25803c19d50386c159848bcb99296f,Landlord Credit,60.180912,0.006293,5.316882,NaN
44802,pyhpc-eos-0-0.0.1,1252ba13daa33b0fbf62293b1df537f3,Landlord Credit,9.554178,0.006708,5.083229,NaN
44829,torch_rnn-3-0.0.1,c5eaf9d4e57252e3bbbf3b34ddf23c35,Landlord Credit,2.646714,0.033546,4.754240,NaN
44866,ckn_resnet50-4-0.0.1,bc23d7d5e46b1fbb5188b50ce93757f6,Landlord Credit,25.662273,0.000000,8.551254,NaN


In [48]:
fqdn_data = fqdn_data[['fqdn', 'gpu_warm', 'gpu_cold']].groupby('fqdn').agg({'gpu_warm': 'mean', 'gpu_cold': 'mean'}).reset_index()

In [49]:
feature_data = pd.read_csv("/Users/akshaykishan/latest_data/processed_data.csv")

In [50]:
[x for x in feature_data.columns if 'gpu' in x]

['gpu_warm_results_sec', 'gpu_cold_results_sec']

In [51]:
fqdn_data_rf = feature_data[['fqdn', 'gpu_warm_results_sec', 'gpu_cold_results_sec']].groupby('fqdn').agg({'gpu_warm_results_sec': 'mean', 'gpu_cold_results_sec': 'mean'}).reset_index()

In [52]:
fqdn_data_rf

,fqdn,gpu_warm_results_sec,gpu_cold_results_sec
0,ckn_mobilenet-0-0.0.1,0.036735,1.126119
1,ckn_mobilenet-1-0.0.1,0.036735,1.126119
2,ckn_mobilenet-2-0.0.1,0.036735,1.126119
3,ckn_mobilenet-3-0.0.1,0.036735,1.126119
4,ckn_mobilenet-4-0.0.1,0.036735,1.126119
...,...,...,...
85,torch_rnn-0-0.0.1,0.032236,0.529059
86,torch_rnn-1-0.0.1,0.032236,0.529059
87,torch_rnn-2-0.0.1,0.032236,0.529059
88,torch_rnn-3-0.0.1,0.032236,0.529059


In [53]:
compare_data = fqdn_data.merge(fqdn_data_rf, on='fqdn', how='inner')

In [54]:
compare_data

,fqdn,gpu_warm,gpu_cold,gpu_warm_results_sec,gpu_cold_results_sec
0,ckn_mobilenet-0-0.0.1,0.000000,3.582151,0.036735,1.126119
1,ckn_mobilenet-1-0.0.1,0.037861,7.884156,0.036735,1.126119
2,ckn_mobilenet-2-0.0.1,0.039049,7.645303,0.036735,1.126119
3,ckn_mobilenet-3-0.0.1,0.000000,5.674562,0.036735,1.126119
4,ckn_mobilenet-4-0.0.1,0.000000,4.643015,0.036735,1.126119
5,ckn_resnet101-0-0.0.1,0.046257,6.908318,0.039543,2.971064
6,ckn_resnet101-1-0.0.1,0.068391,6.963574,0.039543,2.971064
7,ckn_resnet101-2-0.0.1,0.000000,8.559700,0.039543,2.971064
8,ckn_resnet101-3-0.0.1,0.059387,6.806898,0.039543,2.971064
9,ckn_resnet101-4-0.0.1,0.000000,0.000000,0.039543,2.971064
